In [21]:
!uv pip install ultralytics vidgear scikit-learn opencv-python-headless yt-dlp

Checked 5 packages in 43ms


In [22]:
!pip install -U https://github.com/yt-dlp/yt-dlp/archive/master.zip

  Using cached https://github.com/yt-dlp/yt-dlp/archive/master.zip (3.9 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [23]:
import csv
import cv2
import numpy as np
import torch
from datetime import datetime
from ultralytics import YOLO
from sklearn.cluster import KMeans
from collections import Counter, defaultdict

# ==========================================
# 1. Setări de bază
# ==========================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Pregătim motoarele pe: {device.upper()}")

model = YOLO('yolov8m.pt')
model.to(device)

input_video_path = 'meci.mp4'
output_video_path = 'meci_procesat.mp4'

cap = cv2.VideoCapture(input_video_path)

if not cap.isOpened():
    print(f"Eroare: Nu s-a putut deschide fișierul {input_video_path}!")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # ==========================================
    # CSV Logging Setup
    # ==========================================
    CSV_FILE = "raport_fotbal.csv"
    with open(CSV_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Timp", "Numar_Jucatori"])

    LOG_INTERVAL = fps if fps > 0 else 30

    # ==========================================
    # 2. Funcția HSV
    # ==========================================
    def extract_jersey_color_hsv(frame, bbox):
        x1, y1, x2, y2 = map(int, bbox)
        w, h = x2 - x1, y2 - y1
        y_start, y_end = y1 + int(h * 0.15), y1 + int(h * 0.40)
        x_start, x_end = x1 + int(w * 0.25), x2 - int(w * 0.25)
        if y_start >= y_end or x_start >= x_end or y_start < 0 or x_start < 0:
            return None
        roi = frame[y_start:y_end, x_start:x_end]
        if roi.size == 0: return None
        hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        return np.mean(hsv_roi, axis=(0, 1))

    # ==========================================
    # 3. Memorie și Variabile
    # ==========================================
    kmeans = None
    is_kmeans_trained = False
    colors_for_training = []
    team_labels = []
    player_history = defaultdict(list)

    team_colors_draw = {
        0: (0, 0, 255),
        1: (255, 0, 0),
        -1: (150, 150, 150)
    }

    frames_processed = 0
    print(f"Procesare începută! Total cadre: {total_frames}")

    # ==========================================
    # 4. Bucla de Procesare
    # ==========================================
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frames_processed += 1

        results = model.track(frame,
            classes=[0],
            persist=True,
            tracker="bytetrack.yaml",
            conf=0.15,
            iou=0.5,
            imgsz=736,
            half=(device == 'cuda'),
            verbose=False,
            device=device)

        numar_jucatori = len(results[0].boxes) if results[0].boxes is not None else 0

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.cpu().numpy()

            current_frame_colors = []
            valid_indices = []

            for i, box in enumerate(boxes):
                color = extract_jersey_color_hsv(frame, box)
                if color is not None:
                    current_frame_colors.append(color)
                    valid_indices.append(i)

            if not is_kmeans_trained:
                colors_for_training.extend(current_frame_colors)
                if len(colors_for_training) > 50:
                    kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
                    labels = kmeans.fit_predict(colors_for_training)
                    label_counts = Counter(labels)
                    team_labels = [item[0] for item in label_counts.most_common(2)]
                    is_kmeans_trained = True
                    print("\n[INFO] Antrenament KMeans finalizat! Începem desenarea.")
            else:
                if len(current_frame_colors) > 0:
                    instant_labels = kmeans.predict(current_frame_colors)

                    for j, index in enumerate(valid_indices):
                        x1, y1, x2, y2 = map(int, boxes[index])
                        track_id = int(track_ids[index])
                        instant_cluster = instant_labels[j]

                        player_history[track_id].append(instant_cluster)
                        if len(player_history[track_id]) > 45:
                            player_history[track_id].pop(0)

                        stable_cluster = Counter(player_history[track_id]).most_common(1)[0][0]

                        if stable_cluster == team_labels[0]:
                            team_id = 0
                        elif stable_cluster == team_labels[1]:
                            team_id = 1
                        else:
                            team_id = -1

                        draw_color = team_colors_draw[team_id]
                        thickness = 2 if team_id != -1 else 1

                        cv2.rectangle(frame, (x1, y1), (x2, y2), draw_color, thickness)
                        label_text = f"E.{team_id+1}|{track_id}" if team_id != -1 else f"Ref|{track_id}"

                        (text_width, _), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, thickness)
                        cv2.rectangle(frame, (x1, y1 - 20), (x1 + text_width, y1), draw_color, -1)
                        cv2.putText(frame, label_text, (x1, y1 - 5),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # ==========================================
        # 5. HUD — Număr jucători
        # ==========================================
        hud_x, hud_y = 20, 120
        overlay = frame.copy()
        cv2.rectangle(overlay, (hud_x - 10, hud_y - 10), (hud_x + 360, hud_y + 55), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
        cv2.putText(frame, f"Jucatori: {numar_jucatori}", (hud_x, hud_y + 38),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3)

        # ==========================================
        # 6. Salvare CSV o dată pe secundă
        # ==========================================
        if frames_processed % LOG_INTERVAL == 0:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            with open(CSV_FILE, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, numar_jucatori])

        out.write(frame)

        if frames_processed % 10 == 0:
            print(f"Procesat: {frames_processed}/{total_frames} cadre", end="\r")

    # ==========================================
    # 7. Eliberare Resurse
    # ==========================================
    cap.release()
    out.release()
    print(f"\n[SUCCES] Video salvat: {output_video_path}")
    print(f"[SUCCES] Date salvate în: {CSV_FILE}")

Pregătim motoarele pe: CPU
Procesare începută! Total cadre: 8671

[INFO] Antrenament KMeans finalizat! Începem desenarea.
Procesat: 8670/8671 cadre
[SUCCES] Video salvat: meci_procesat.mp4
[SUCCES] Date salvate în: raport_fotbal.csv
